In [1]:
# 导包
import torch
import torch.nn.functional as F # 导入神经网络函数库

In [2]:
# 数据准备

n_item = 1000   # 样本容量m
n_feature = 2   # 特征维度n，特征的个数
learning_rate = 0.001   # 学习率，步长
epochs = 100    # 训练轮数

# fake data: 构造假数据
torch.manual_seed(123) # 指定任意随机种子构造随机数
data_x = torch.randn(size=(n_item, n_feature)).float() # 依据随机种子对象生成1000行，2个特征的数据集，取值给定浮点数
data_y = torch.where(torch.subtract(data_x[:, 0]*0.5, data_x[:, 1]*1.5) > 0, 1., 0.).float()
# 这里标签的构建规则是：如果第一个数的0.5倍大于第二个数的0.5倍，则该样本标签取值为1，否则为0

In [4]:
# 借助Pytorch手写逻辑回归算法
class LogisticRegressionManually(object):
    def __init__(self):
        # regression model
        self.w = torch.randn(size=(n_feature, 1), requires_grad=True) # 构造待估参数列向量w
        self.b = torch.zeros(size=(1, 1), requires_grad=True) # 构造偏置b
        # requires_grad为true表示当前这个量是可以求导的，在这里就表示可以进行梯度更新
        # 如果不指定就说明这个量是不能求导更新的
    
    # 定义前向计算，说白了就是构建逻辑回归的决策函数，计算得出标签结果
    def forward(self, x):
        y_hat = F.sigmoid(torch.matmul(self.w.transpose(0, 1), x) + self.b) # 这里其实就是在套用逻辑回归的决策函数w^Tx + b，转置
        return y_hat # 返回预测结果
    
    # 定义损失函数，
    @staticmethod
    def loss_func(y_hat, y):
        return -(torch.log(y_hat)*y + (1-y)*torch.log(1-y_hat)) # 这里其实就是在套用逻辑回归的损失函数，也是手写了一个二分类交叉熵损失函数

    # train，构建训练过程
    def train(self):
        # 100轮训练过程，观察损失函数的下降规律
        for epoch in range(epochs):
            # 1. load data，加载每一条样本数据
            for step in range(n_item):
                # 2. forward calc，前向计算，得出标签预测结果
                y_hat = self.forward(data_x[step])
                y = data_y[step]    # target，在获取当前样本的真实标签结果
                # 3. loss calc，损失函数计算，传入预测值和真实值
                loss = self.loss_func(y_hat, y)
                # 4. backward
                loss.backward() # 反向求导
                # 5. update model(update param)
                with torch.no_grad(): # 更新模型参数
                # 这里为啥要使用with语句，表示在这个上下文环境下不执行梯度更新操作，此时需要完成更新参数
                    self.w.data -= learning_rate * self.w.grad.data # 待估参数列向量w更新
                    self.b.data -= learning_rate * self.b.grad.data # 截距b更新
                self.w.grad.data.zero_() # 记得更新完后归零
                self.b.grad.data.zero_()
            if epoch % 10 == 0:
                print('Epoch: {}, Loss: {}'.format(epoch, loss.item()))


In [5]:
# 执行
if __name__ == '__main__':
    lrm = LogisticRegressionManually()  # 新建模型
    lrm.train()  # 训练

Epoch: 0, Loss: 0.5620617270469666
Epoch: 10, Loss: 0.4762253761291504
Epoch: 20, Loss: 0.43637001514434814
Epoch: 30, Loss: 0.4089241325855255
Epoch: 40, Loss: 0.3875424861907959
Epoch: 50, Loss: 0.3698981702327728
Epoch: 60, Loss: 0.35483989119529724
Epoch: 70, Loss: 0.3416937291622162
Epoch: 80, Loss: 0.33002859354019165
Epoch: 90, Loss: 0.3195469379425049


In [16]:
# 导包
import torch
from torch import nn # 神经网络
from torch.nn import functional as F # 工具库

In [17]:
# 指定device是GPU还是CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [18]:
# fake data，构建假数据
n_item = 1000 # 1000条样本
n_feature = 2 # 2个特征

torch.manual_seed(123) # 指定随机种子
data_x = torch.randn(size=(n_item, n_feature)).float() # 构造数据集
data_y = torch.where(torch.subtract(data_x[:, 0]*0.5, data_x[:, 1]*1.5)+0.02 > 0, 0, 1).long() # 构造标签

data_y = F.one_hot(data_y)  # one hot encode，标签完成独热编码
""" shape: [n_item, 2]
tensor([[1, 0],
        [1, 0],
        [0, 1],
        ...,
        [1, 0],
        [1, 0],
        [1, 0]])
"""

' shape: [n_item, 2]\ntensor([[1, 0],\n        [1, 0],\n        [0, 1],\n        ...,\n        [1, 0],\n        [1, 0],\n        [1, 0]])\n'

In [19]:
# 构建二分类模型
class BinaryClassificationModel(nn.Module):
    def __init__(self, in_feature):
        super(BinaryClassificationModel, self).__init__()
        """ single perceptron 实现单层感知机
        self.layer_1 = nn.Linear(in_features=in_feature, out_features=2, bias=True) # 指定数据的维度，输出的维度，是否学习偏置
        """

        """ multi perceptron 实现多层感知机"""
        self.layer_1 = nn.Linear(in_features=in_feature, out_features=128, bias=True) # 指定输入的神经元个数，输出的神经元个数，是否包含偏置
        self.layer_2 = nn.Linear(in_features=128, out_features=512, bias=True) # 第二层输入与第一层输出数量保持一致
        # 。。。
        self.layer_final = nn.Linear(in_features=512, out_features=2, bias=True) # 输出层，二分类，只有两个结果
    
    # 前向计算
    def forward(self, x):
        layer_1_output = F.sigmoid(self.layer_1(x)) # 直接指定sigmoid激活函数
        layer_2_output = F.sigmoid(self.layer_2(layer_1_output))
        output = F.sigmoid(self.layer_final(layer_2_output))
        return output

In [20]:
# hyper parameters 超参数初始化值
learning_rate = 0.01 # 学习率，步长
epochs = 100 # 训练的轮数

In [21]:
# 构建模型
model = BinaryClassificationModel(n_feature).to(device)

In [22]:
# 指定优化器
opt = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [23]:
# 构建损失函数 - 二分类交叉熵损失函数
criteria = nn.BCELoss()

In [25]:
# train loop 执行训练过程
for epoch in range(epochs): # 训练轮数
    for step in range(n_item):
        x = data_x[step].to(device)
        y = data_y[step].to(device)

        opt.zero_grad() # 梯度归零

        y_hat = model(x.unsqueeze(0)) # [1, 2] # 模型训练获取预测结果
        # [1, 2]: [[0.9, 0.1]]
        loss = criteria(y_hat, y.unsqueeze(0).float()) # 指定损失函数
        loss.backward() # 梯度下降，反向求导
        opt.step()
    if epoch % 10 == 0:
        print('Epoch: %03d, Loss: %.3f' % (epoch, loss.cpu().item()))


Epoch: 000, Loss: 0.323
Epoch: 010, Loss: 0.136
Epoch: 020, Loss: 0.055
Epoch: 030, Loss: 0.026
Epoch: 040, Loss: 0.014
Epoch: 050, Loss: 0.008
Epoch: 060, Loss: 0.005
Epoch: 070, Loss: 0.003
Epoch: 080, Loss: 0.002
Epoch: 090, Loss: 0.001
